# 🌍 AfricaInvest Intelligence — Analyse Exploratoire (EDA)

**Objectif** : Explorer le dataset d'indicateurs économiques africains et comprendre les patterns liés à la croissance économique.

| # | Section |
|---|--------|
| 1 | Chargement & aperçu général |
| 2 | Distribution de la variable cible |
| 3 | Analyse des features numériques |
| 4 | Analyse géographique (régions & pays) |
| 5 | Corrélations & relations inter-variables |
| 6 | Analyse temporelle |
| 7 | Synthèse & recommandations |

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
sns.set_theme(style='darkgrid', palette='viridis')

DATA_PATH = Path('../data/africa_economic_data.csv')
print('Environnement prêt ✅')

---
## 1. Chargement & aperçu général

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Dataset : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')
print(f'Pays    : {df["country"].nunique()} | Régions : {df["region"].nunique()}')
print(f'Années  : {df["year"].min()} – {df["year"].max()}')
print(f'\nValeurs manquantes :\n{df.isnull().sum()}')
df.head(8)

In [ ]:
NUMERIC_COLS = [
    'gdp_growth', 'inflation_rate', 'unemployment_rate', 'fdi_pct_gdp',
    'trade_openness', 'literacy_rate', 'population_growth',
    'internet_penetration', 'government_debt_pct', 'natural_resources_rents',
]
display(df[NUMERIC_COLS].describe().round(2))

---
## 2. Distribution de la variable cible

In [ ]:
COLORS = {'high': '#56d364', 'medium': '#e3b341', 'low': '#f85149'}

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
counts = df['growth_category'].value_counts()

axes[0].bar(counts.index, counts.values,
            color=[COLORS[k] for k in counts.index], edgecolor='#0d1117', linewidth=1.5)
axes[0].set_title('Distribution des classes de croissance', fontweight='bold')
axes[0].set_xlabel('Catégorie')
axes[0].set_ylabel('Nombre d\'observations')
for i, (k, v) in enumerate(counts.items()):
    axes[0].text(i, v + 8, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=9)

axes[1].pie(counts.values, labels=counts.index,
            colors=[COLORS[k] for k in counts.index],
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': '#0d1117', 'linewidth': 2})
axes[1].set_title('Répartition (%)', fontweight='bold')

plt.tight_layout()
plt.savefig('../artifacts/eda_target_distribution.png', bbox_inches='tight', facecolor='white')
plt.show()

---
## 3. Analyse des features numériques

In [ ]:
# Distributions par feature
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, col in enumerate(NUMERIC_COLS):
    for cat, color in COLORS.items():
        subset = df[df['growth_category'] == cat][col]
        axes[i].hist(subset, bins=25, alpha=0.55, label=cat, color=color, edgecolor='none')
    axes[i].set_title(col.replace('_', ' ').title(), fontweight='bold', fontsize=9)
    axes[i].set_xlabel('')
    if i == 0:
        axes[i].legend(fontsize=8)

plt.suptitle('Distribution des features par classe de croissance', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../artifacts/eda_feature_distributions.png', bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# Boxplots par catégorie
KEY_FEATURES = ['gdp_growth', 'inflation_rate', 'fdi_pct_gdp', 'literacy_rate', 'internet_penetration']
order = ['low', 'medium', 'high']
palette = {'low': '#f85149', 'medium': '#e3b341', 'high': '#56d364'}

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
for ax, col in zip(axes, KEY_FEATURES):
    sns.boxplot(data=df, x='growth_category', y=col, order=order,
                palette=palette, ax=ax, linewidth=1.2)
    ax.set_title(col.replace('_', ' ').title(), fontweight='bold')
    ax.set_xlabel('')

plt.suptitle('Distribution des features clés par classe (boxplots)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../artifacts/eda_boxplots.png', bbox_inches='tight', facecolor='white')
plt.show()

---
## 4. Analyse géographique

In [ ]:
# Croissance par région
region_growth = df.groupby(['region', 'growth_category']).size().unstack(fill_value=0)
region_growth_pct = region_growth.div(region_growth.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

region_growth.plot(kind='bar', ax=axes[0], color=[COLORS[c] for c in region_growth.columns],
                   edgecolor='white', linewidth=0.5)
axes[0].set_title('Observations par région & classe', fontweight='bold')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=20)
axes[0].legend(title='Classe')

region_growth_pct.plot(kind='bar', stacked=True, ax=axes[1],
                        color=[COLORS[c] for c in region_growth_pct.columns],
                        edgecolor='white', linewidth=0.5)
axes[1].set_title('Répartition (%) par région', fontweight='bold')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=20)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].legend(title='Classe')

plt.tight_layout()
plt.savefig('../artifacts/eda_regional_analysis.png', bbox_inches='tight', facecolor='white')
plt.show()

---
## 5. Matrice de corrélations

In [ ]:
corr = df[NUMERIC_COLS].corr()

fig, ax = plt.subplots(figsize=(11, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1, ax=ax,
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
ax.set_title('Matrice de corrélation — Indicateurs économiques', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('../artifacts/eda_correlation_matrix.png', bbox_inches='tight', facecolor='white')
plt.show()

---
## 6. Analyse temporelle

In [ ]:
yearly = df.groupby(['year', 'growth_category']).size().unstack(fill_value=0)
yearly_pct = yearly.div(yearly.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(15, 4))

for cat, color in COLORS.items():
    if cat in yearly.columns:
        axes[0].plot(yearly.index, yearly[cat], color=color, label=cat, linewidth=2)
axes[0].set_title('Nombre d\'observations par année & classe', fontweight='bold')
axes[0].set_xlabel('Année')
axes[0].legend()

for cat, color in COLORS.items():
    if cat in yearly_pct.columns:
        axes[1].plot(yearly_pct.index, yearly_pct[cat], color=color, label=cat, linewidth=2)
axes[1].set_title('Proportion (%) par classe au fil du temps', fontweight='bold')
axes[1].set_xlabel('Année')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].legend()

plt.tight_layout()
plt.savefig('../artifacts/eda_temporal_analysis.png', bbox_inches='tight', facecolor='white')
plt.show()

---
## 7. Synthèse & Recommandations

### 🔑 Observations clés

| Feature | Relation avec `high growth` |
|---------|-----------------------------|
| `gdp_growth` | **Forte corrélation positive** — logique (feature directement liée à la cible) |
| `fdi_pct_gdp` | Corrélation positive — les IDE stimulent la croissance |
| `literacy_rate` | Corrélation positive — capital humain |
| `internet_penetration` | Corrélation positive — indicateur de modernisation |
| `inflation_rate` | Corrélation négative — instabilité macro |
| `unemployment_rate` | Corrélation négative — faible utilisation des ressources |

### ⚙️ Recommandations pour le modèle

1. **Classe `low` sous-représentée** → utiliser `class_weight='balanced'` dans le classificateur.
2. **`inflation_rate` et `natural_resources_rents`** ont des distributions très asymétriques → envisager une transformation log.
3. **Potentiel de feature engineering** : ratios (ex: `fdi_pct_gdp / unemployment_rate`), interaction `literacy × internet`.
4. **Dimension temporelle** : intégrer `year` comme feature ou construire des lags si données temporelles réelles.